# Projeto 2: Clustering de Dados e Detecção de Comunidades em Grafos

**Dataset:** Iris

**Integrantes:**

1. Anna Beatriz Souza e Silva - 13694103
2. Milena Rodrigues Monteiro - 12566157


## Etapa 1 - Preparação dos dados

O dataset Iris foi importado do **UC Irvine Machine Learning Repository**. Suas demais descrições e instruções para importar os dados em Python estão presentes [nesse link](https://archive.ics.uci.edu/dataset/53/iris).

Nesse dataset, cada instância corresponde a uma planta, e têm variáveis representando certas descrições de cada uma com um valor contínuo em centímetros.

### Instalando bibliotecas necessárias

In [ ]:
%pip install ucimlrepo scikit-learn networkx matplotlib pandas

### Importando os dados

In [5]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# fetch dataset
iris = fetch_ucirepo(id=53)

# dados (em formato de pandas dataframes)
X = iris.data.features
y = iris.data.targets

### Amostra dos dados

Nessa parte vamos entender os dados que veem no dataset e ver quais são as classes contidas nele.

In [31]:
display(X.sample(5))

,sepal length,sepal width,petal length,petal width
13,4.3,3.0,1.1,0.1
40,5.0,3.5,1.3,0.3
126,6.2,2.8,4.8,1.8
51,6.4,3.2,4.5,1.5
85,6.0,3.4,4.5,1.6


In [33]:
display(y.sample(5))

,class
13,Iris-setosa
106,Iris-virginica
71,Iris-versicolor
84,Iris-versicolor
117,Iris-virginica


In [37]:
# Mostrando todas as classes únicas possíveis do dataset
display(pd.DataFrame({"class": y["class"].unique()}))

,class
0,Iris-setosa
1,Iris-versicolor
2,Iris-virginica


### Análise das classes em `y`

Como visto na célula acima, existem **3 classes possíveis** em `y`:
- Iris-setosa
- Iris-versicolor
- Iris-virginica

Como elas são as classes categóricas, **elas não serão padronizadas ou normalizadas**, já que representam o *ground truth* utilizado quando avaliarmos os agrupamentos.

### Padronização dos atributos em `X`

Além disso, temos as seguintes **colunas** em $X$:
- sepal length
- sepal width
- petal length
- petal width

Como o restante do trabalho calcula distâncias, se uma coluna tiver maior escala ou variação, ela pode dominar o resultado. A padronização evita isso.

Isso significa que teremos que padronizar os valores das colunas de `X` utilizando a padronização:

$$
z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

onde:

- $\mu_j$ é a média do atributo $j$;
- $\sigma_j$ é o desvio-padrão do atributo $j$.

In [40]:
# Média e desvio-padrão de cada coluna
media = X.mean()
desvio_padrao = X.std()

# Padronização: z = (x - média) / desvio-padrão
X = (X - media) / desvio_padrao

display(X.head())

,sepal length,sepal width,petal length,petal width
0,-0.897674,1.028611,-1.336794,-1.308593
1,-1.139200,-0.124540,-1.336794,-1.308593
2,-1.380727,0.336720,-1.393470,-1.308593
3,-1.501490,0.106090,-1.280118,-1.308593
4,-1.018437,1.259242,-1.336794,-1.308593


## Etapa 2 - Clustering tradicional (Abordagem 1)

Como pontos em um espaço de atributos, utilizando um algoritmo tradicional de clustering.

$$
X \rightarrow C_{\text{cluster}}
$$


### Aplicação do K-Means

O algoritmo recomendado foi o **K-Means**. O objetivo do K-Means é minimizar a soma das distâncias quadráticas entre cada objeto e o centroide do seu cluster:

$$
J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2
$$

O centroide do cluster $C_k$ é calculado por:

$$
\mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i
$$


### 1.2. Justificação a escolha do número de clusters $K$.


### 1.3. Construção do grafo de similaridade

Os mesmos objetos deverão ser representados por um grafo:

$$
G = (V, E)
$$

Cada vértice $v_i \in V$ representa um objeto $x_i$.

A distância entre dois objetos poderá ser calculada utilizando a distância Euclidiana:

$$
d(x_i, x_j) =
\sqrt{
\sum_{l=1}^{d}
(x_{il} - x_{jl})^2
}
$$

A partir dessas distâncias deverá ser construído um **k-Nearest Neighbors Graph (k-NN Graph)**.

Cada objeto será conectado aos seus $k$ vizinhos mais próximos.


### 1.4. Pesos das arestas

Opcionalmente, o grafo poderá ser ponderado.

Uma função de similaridade Gaussiana pode ser utilizada:

$$
w_{ij} =
\exp\left(
-\frac{d(x_i,x_j)^2}{2\sigma^2}
\right)
$$

Dessa forma, objetos muito semelhantes terão pesos de conexão maiores.


### 1.5. Comparação entre clustering e detecção de comunidades

Os alunos deverão comparar a partição obtida pelo clustering:

$$
C_{\text{cluster}}
$$

com a partição obtida pela detecção de comunidades:

$$
C_{\text{community}}
$$

Uma das métricas recomendadas é o **Adjusted Rand Index (ARI)**:

$$
ARI =
\frac{
\displaystyle
\sum_{ij}\binom{n_{ij}}{2}
-
\frac{
\displaystyle
\sum_i\binom{a_i}{2}
\sum_j\binom{b_j}{2}
}{
\displaystyle
\binom{n}{2}
}
}{
\displaystyle
\frac{1}{2}
\left[
\sum_i\binom{a_i}{2}
+
\sum_j\binom{b_j}{2}
\right]
-
\frac{
\displaystyle
\sum_i\binom{a_i}{2}
\sum_j\binom{b_j}{2}
}{
\displaystyle
\binom{n}{2}
}
}
$$

onde:

- $n$ = número total de objetos;
- $n_{ij}$ = número de objetos que pertencem simultaneamente ao cluster $i$ da primeira partição e ao cluster $j$ da segunda partição;
- $a_i = \sum_j n_{ij}$ = tamanho do cluster $i$ da primeira partição;
- $b_j = \sum_i n_{ij}$ = tamanho do cluster $j$ da segunda partição;
- $\binom{n}{2}$ = número total de pares de objetos.

### Interpretação do ARI

- **ARI = 1** → as duas partições são idênticas;
- **ARI ≈ 0** → a concordância é aproximadamente a esperada ao acaso;
- **ARI < 0** → a concordância é inferior à esperada ao acaso.

Outra possibilidade é utilizar o **Normalized Mutual Information (NMI)**:

$$
NMI(U,V) =
\frac{2I(U;V)}{H(U)+H(V)}
$$

onde:

- $I(U;V)$ representa a informação mútua entre as duas partições;
- $H(U)$ e $H(V)$ representam suas respectivas entropias.

A fórmula da **Informação Mútua (Mutual Information — MI)** entre duas partições $U$ e $V$ é:

$$
I(U;V) =
\sum_{i=1}^{|U|}
\sum_{j=1}^{|V|}
p(u_i,v_j)
\log
\left(
\frac{p(u_i,v_j)}
{p(u_i)p(v_j)}
\right)
$$

onde:

- $U = \{u_1, u_2, \ldots, u_{|U|}\}$ representa a primeira partição;
- $V = \{v_1, v_2, \ldots, v_{|V|}\}$ representa a segunda partição;
- $p(u_i,v_j)$ é a probabilidade de um objeto pertencer simultaneamente aos grupos $u_i$ e $v_j$;
- $p(u_i)$ é a probabilidade de um objeto pertencer ao grupo $u_i$;
- $p(v_j)$ é a probabilidade de um objeto pertencer ao grupo $v_j$.

Os alunos deverão utilizar **pelo menos uma dessas métricas** para avaliar a similaridade entre os dois agrupamentos.


### 1.6. Experimento com diferentes valores de $k$

Os alunos deverão investigar como o valor de $k$ utilizado na construção do grafo influencia o resultado da detecção de comunidades.

Para cada valor de $k$, deverão registrar:

| $k$ | Nº de comunidades | Modularidade $Q$ | NMI/ARI |
| --: | ----------------: | ---------------: | ------: |
|   3 |                 — |                — |       — |
|   5 |                 — |                — |       — |
|  10 |                 — |                — |       — |
|  15 |                 — |                — |       — |
|  20 |                 — |                — |       — |

Os resultados deverão ser apresentados também por meio de gráficos.

Os alunos deverão discutir:

- Como o aumento de $k$ altera a densidade do grafo?
- Como o número de comunidades muda?
- Qual valor de $k$ produz os melhores resultados?
- O que acontece quando $k$ é muito pequeno?
- O que acontece quando $k$ é muito grande?
